# 08 — Hugging Face Transformers: Tokenizers, Models, Fine-Tuning, Generation, and Chat Patterns

Goal: practical modern NLP/LLM workflows using PyTorch + Transformers + Accelerate.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. HF quickstart (templates)

These snippets typically require internet to download checkpoints. If offline, point to a local path.

In [ ]:

hf_quickstart = r'''
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)

prompt = "Hello, my name is"
inputs = tokenizer(prompt, return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=0.8, top_p=0.95)
print(tokenizer.decode(out[0], skip_special_tokens=True))
'''
print(hf_quickstart)

## 2. Fine-tuning a classifier with Trainer (template)

In [ ]:

hf_trainer_template = r'''
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tok(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

tokenized = dataset.map(tok, batched=True)
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label","labels")
tokenized.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
acc = evaluate.load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return acc.compute(predictions=preds, references=p.label_ids)

args = TrainingArguments(
    output_dir="out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()
'''
print(hf_trainer_template[:800] + "\n...\n")

## 3. Chat prompt formatting

Chat models define their own templates (system/user/assistant). Many tokenizers include `.apply_chat_template`.
If unavailable, you must follow the model card prompt rules exactly.

In [ ]:

chat_template = r'''
history = []
def build_prompt(history):
    prompt = ""
    for role, text in history:
        if role == "user":
            prompt += f"User: {text}\n"
        else:
            prompt += f"Assistant: {text}\n"
    prompt += "Assistant: "
    return prompt
'''
print(chat_template)

## 4. Accelerate (concept)

Accelerate manages:
- device placement
- mixed precision
- DDP/FSDP setup
- gradient accumulation

Use it when training medium-to-large models.